Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn
!pip install pandas
!pip install tqdm
!pip install scikit-image
!pip install scipy
!pip install ace_tools
!pip install imbalanced-learn

Calling the Libraries:

In [ ]:
from skimage.feature import local_binary_pattern
from skimage.color import rgb2gray
from skimage import exposure
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import local_binary_pattern
from skimage.color import rgb2gray
from skimage import exposure
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import cv2
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE
from collections import Counter
import glob
import os
from sklearn.preprocessing import normalize
import cv2
import glob
import os

Finding Height and Width of an Image:

In [ ]:
import cv2
import os

# Example sample image path from session 1
sample_image_path = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein/vein001_1/01.jpg'

# Load the image in grayscale
img = cv2.imread(sample_image_path, cv2.IMREAD_GRAYSCALE)

# Check if image was loaded successfully
if img is None:
    print("Image could not be loaded. Check the path.")
else:
    # Print its shape
    print("Image shape:", img.shape)

    # Print height and width
    height, width = img.shape
    print("Height:", height)
    print("Width:", width)

Train:

In [ ]:
import os
import cv2
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE
from collections import Counter
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import normalize
import pandas as pd

# ----------------------------
# CONFIGURATION
# ----------------------------
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'

NUM_SUBJECTS = 123
NUM_FINGERS = 4
IMAGE_SIZE = (100, 300)
PATCH_SIZE = 10
stride = 10
#LBP_CONFIGS = [(1, 16), (1, 8), (2, 8)]  # (radius, points)
LBP_CONFIGS = [(2, 8)]
# ----------------------------
# FUNCTION: RIU2 Mapping
# ----------------------------
def get_riu2_mapping(P):
    table = np.zeros(2 ** P, dtype=np.uint8)
    for i in range(2 ** P):
        binary = [(i >> j) & 1 for j in range(P)]
        rotations = [binary[n:] + binary[:n] for n in range(P)]
        min_rotation = min(rotations)
        extended = min_rotation + [min_rotation[0]]
        transitions = sum(extended[j] != extended[j + 1] for j in range(P))
        if transitions <= 2:
            table[i] = sum(min_rotation)
        else:
            table[i] = P + 1
    return table

# ----------------------------
# FUNCTION: Extract LBP histogram
# ----------------------------
def extract_lbp_histogram(block, P, R, riu2_map):
    lbp = local_binary_pattern(block, P, R, method='ror').astype(np.uint16)
    lbp_mapped = riu2_map[lbp]
    hist, _ = np.histogram(
        lbp_mapped.ravel(),
        bins=np.arange(0, P + 3),
        density=True
    )
    return hist

# ----------------------------
# PRE-COMPUTE MAPPINGS
# ----------------------------
mapping_dict = {P: get_riu2_mapping(P) for _, P in LBP_CONFIGS}

# ----------------------------
# MAIN LOOP (Training: Strategy 1 - Protocol 1)
# ----------------------------
train_lbp_features = []
train_labels = []

print("\U0001F680 Extracting training features (Strategy 1 - Protocol 1)...")

for subject_id in tqdm(range(1, NUM_SUBJECTS + 1)):
    for base_path in [base_path_sess1, base_path_sess2]:
        session = 's1' if '1st_session' in base_path else 's2'

        for img_idx in [1, 2, 3, 4, 5]:  # Training images
            fused_vector = []
            print(f"\n➡️ Subject {subject_id:03d}, Session: {session}, Image: {img_idx} (Training)")

            for finger_id in range(1, NUM_FINGERS + 1):
                folder = f"vein{subject_id:03d}_{finger_id}"
                img_path = os.path.join(base_path, folder, f"{img_idx:02d}.jpg")
                print(f"  📥 Loading finger {finger_id} from: {img_path}")

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"  ❌ Missing image: {img_path}")
                    continue

                img = cv2.resize(img, IMAGE_SIZE)
                img = cv2.fastNlMeansDenoising(img, h=10)
                img = cv2.equalizeHist(img).astype(np.float64) / 255.0
                img = (img - np.mean(img)) / (np.std(img) + 1e-8)

                for y in range(0, IMAGE_SIZE[1] - PATCH_SIZE + 1, stride):
                    for x in range(0, IMAGE_SIZE[0] - PATCH_SIZE + 1, stride):
                        block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                        hist = []
                        for R, P in LBP_CONFIGS:
                            mapped = extract_lbp_histogram(block, P, R, mapping_dict[P])
                            hist.extend(mapped)
                        fused_vector.extend(hist)

            if len(fused_vector) > 0:
                train_lbp_features.append(fused_vector)
                train_labels.append(f"{subject_id:03d}_img{img_idx}_s{session}")

# ----------------------------
# NORMALIZE AND FINALIZE
# ----------------------------
train_lbp_features = np.array(train_lbp_features, dtype=np.float32)
train_lbp_features = normalize(train_lbp_features, norm='l2')
train_labels = np.array(train_labels)

print("\n✅ Training feature extraction complete!")
print("🔢 Feature matrix shape:", train_lbp_features.shape)
print("🟢 Example labels:", train_labels[:5])

Test:

In [ ]:
test_lbp_features = []
test_labels = []

print("🧪 Extracting testing features (Strategy 1 - Protocol 1)...")

for subject_id in tqdm(range(1, NUM_SUBJECTS + 1)):
    for base_path in [base_path_sess1, base_path_sess2]:
        session = 's1' if '1st_session' in base_path else 's2'

        for img_idx in [6]:  # Testing images
            fused_vector = []
            print(f"\n➡️ Subject {subject_id:03d}, Session: {session}, Image: {img_idx} (Testing)")

            for finger_id in range(1, NUM_FINGERS + 1):
                folder = f"vein{subject_id:03d}_{finger_id}"
                img_path = os.path.join(base_path, folder, f"{img_idx:02d}.jpg")
                print(f"  📥 Loading finger {finger_id} from: {img_path}")

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"  ❌ Missing image: {img_path}")
                    continue

                img = cv2.resize(img, IMAGE_SIZE)
                img = cv2.fastNlMeansDenoising(img, h=10)
                img = cv2.equalizeHist(img).astype(np.float64) / 255.0
                img = (img - np.mean(img)) / (np.std(img) + 1e-8)

                for y in range(0, IMAGE_SIZE[1] - PATCH_SIZE + 1, stride):
                    for x in range(0, IMAGE_SIZE[0] - PATCH_SIZE + 1, stride):
                        block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                        hist = []
                        for R, P in LBP_CONFIGS:
                            mapped = extract_lbp_histogram(block, P, R, mapping_dict[P])
                            hist.extend(mapped)
                        fused_vector.extend(hist)

            if len(fused_vector) > 0:
                test_lbp_features.append(fused_vector)
                test_labels.append(f"{subject_id:03d}_img{img_idx}_s{session}")

test_lbp_features = np.array(test_lbp_features, dtype=np.float32)
test_lbp_features = normalize(test_lbp_features, norm='l2')
test_labels = np.array(test_labels)

print("✅ Testing data shape:", test_lbp_features.shape)


Benchmarking

In [ ]:
# Helper function
def extract_subject_and_session(label):
    parts = label.split('_')
    subject = parts[0]  # e.g., "003"
    session = parts[-1]  # e.g., "s1"
    return subject, session

correct_matches = 0
total_tests = len(test_lbp_features)

print("\n🔍 Classifying test data using Manhattan distance (Match: Subject ID + Session)...\n")

for i in range(total_tests):
    test_vec = test_lbp_features[i]
    true_label = test_labels[i]

    distances = np.sum(np.abs(train_lbp_features - test_vec), axis=1)
    min_index = np.argmin(distances)
    predicted_label = train_labels[min_index]

    true_subj, true_sess = extract_subject_and_session(true_label)
    pred_subj, pred_sess = extract_subject_and_session(predicted_label)

    if true_subj == pred_subj and true_sess == pred_sess:
        correct_matches += 1
        match_symbol = "✅"
    else:
        match_symbol = "❌"

    print(f"Test sample {i+1}: Predicted = {predicted_label}, Actual = {true_label} {match_symbol}")

accuracy = (correct_matches / total_tests) * 100
print("\n📊 Final Results")
print(f"✅ Correct matches: {correct_matches} / {total_tests}")
print(f"🎯 Recognition Accuracy: {accuracy:.2f}%")


Benchmarking 2:

In [ ]:
import numpy as np

# ✅ Helper function to extract subject ID only (ignore finger and session)
def extract_subject_id(label):
    parts = label.split('_')
    subject = parts[0]  # e.g., "008" from "008_img6_ss2"
    return subject

correct_matches = 0
total_tests = len(test_lbp_features)

print("\n🔍 Classifying test data using Manhattan distance (Match: Subject ID only)...\n")

for i in range(total_tests):
    test_vec = test_lbp_features[i]
    true_label = test_labels[i]

    # Manhattan distances to all training vectors
    distances = np.sum(np.abs(train_lbp_features - test_vec), axis=1)
    min_index = np.argmin(distances)
    predicted_label = train_labels[min_index]

    # Extract subject ID only
    true_subj = extract_subject_id(true_label)
    pred_subj = extract_subject_id(predicted_label)

    if true_subj == pred_subj:
        correct_matches += 1
        match_symbol = "✅"
    else:
        match_symbol = "❌"

    print(f"Test sample {i+1}: Predicted = {predicted_label}, Actual = {true_label} {match_symbol}")

# 📊 Final Accuracy
accuracy = (correct_matches / total_tests) * 100
print("\n📊 Final Results")
print(f"✅ Correct person matches: {correct_matches} / {total_tests}")
print(f"🎯 Recognition Accuracy (Subject Only): {accuracy:.2f}%")


Session Sensitive R5

In [ ]:
import numpy as np
from collections import defaultdict

# === Configuration ===
ranks = [1, 5]
rank_correct = defaultdict(int)
total_tests = len(test_labels)

print("📊 Calculating Session-Sensitive CMC (Rank-1 & Rank-5) — Strategy 1: Fused Fingers...")

for i in range(total_tests):
    proj_test = test_lbp_features[i]
    test_label = test_labels[i]

    # Extract subject and session (no finger)
    test_parts = test_label.split('_')
    test_subject = test_parts[0]
    test_session = test_parts[-1]
    test_id = f"{test_subject}_{test_session}"

    # Compute Manhattan distances
    distances = np.sum(np.abs(train_lbp_features - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    # Search for the first correct match
    matched = False
    for r in range(1, max(ranks) + 1):
        candidate_label = train_labels[sorted_indices[r - 1]]
        parts = candidate_label.split('_')
        candidate_subject = parts[0]
        candidate_session = parts[-1]
        candidate_id = f"{candidate_subject}_{candidate_session}"

        if candidate_id == test_id and not matched:
            for k in ranks:
                if r <= k:
                    rank_correct[k] += 1
            matched = True

# === Final CMC Results
for k in ranks:
    accuracy = (rank_correct[k] / total_tests) * 100
    print(f"🎯 Rank-{k} Accuracy (Subject + Session): {accuracy:.2f}%")


Session Sensitive CMC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === CONFIGURATION ===
max_rank = 100
rank_correct = np.zeros(max_rank)
total_tests = len(test_lbp_features)

print("📊 Calculating Session-Sensitive CMC Curve (Rank-1 to Rank-100)...")

for i in range(total_tests):
    proj_test = test_lbp_features[i]
    test_parts = test_labels[i].split('_')
    true_subject = test_parts[0]
    true_session = test_parts[-1]
    true_id = f"{true_subject}_{true_session}"  # Strategy 1: Subject + Session only

    distances = np.sum(np.abs(train_lbp_features - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    for r in range(max_rank):
        candidate_label = train_labels[sorted_indices[r]]
        cand_parts = candidate_label.split('_')
        cand_subject = cand_parts[0]
        cand_session = cand_parts[-1]
        candidate_id = f"{cand_subject}_{cand_session}"

        if candidate_id == true_id:
            rank_correct[r:] += 1
            break

# === Normalize to percentage
cmc_curve = (rank_correct / total_tests) * 100

# === Plotting the CMC Curve
plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, max_rank + 1), cmc_curve, label="Session-Sensitive CMC", linewidth=2)
plt.xlabel("Rank")
plt.ylabel("Identification Accuracy (%)")
plt.title("Session-Sensitive CMC Curve — LBP$_{\\mathrm{RIU2}}$((8,1), (16,1), (8,2)) (Strategy 1, Protocol 1)")
plt.grid(True)
plt.xticks(np.arange(0, max_rank + 1, 10))
plt.legend()
plt.tight_layout()
plt.show()

# === Print Key Rank Accuracies
print(f"🎯 Rank-1 Accuracy   : {cmc_curve[0]:.2f}%")
print(f"🎯 Rank-5 Accuracy   : {cmc_curve[4]:.2f}%")
print(f"🎯 Rank-10 Accuracy  : {cmc_curve[9]:.2f}%")
print(f"🎯 Rank-100 Accuracy : {cmc_curve[99]:.2f}%")


Session Sensitive Precision, Recall, F1, Accuracy

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# === Initialize lists
all_scores = []
all_labels = []

# === Pairwise score computation for Session-Sensitive Verification (Strategy 1: Fused Fingers)
for test_idx in range(len(test_lbp_features)):
    test_vec = test_lbp_features[test_idx]
    test_label = test_labels[test_idx]
    test_parts = test_label.split('_')
    test_subject = test_parts[0]
    test_session = test_parts[-1]
    test_id = f"{test_subject}_{test_session}"  # Strategy 1: Match on subject + session only

    for train_idx in range(len(train_lbp_features)):
        train_vec = train_lbp_features[train_idx]
        train_label = train_labels[train_idx]
        train_parts = train_label.split('_')
        train_subject = train_parts[0]
        train_session = train_parts[-1]
        train_id = f"{train_subject}_{train_session}"

        # Similarity score: negative Manhattan distance
        score = -np.sum(np.abs(test_vec - train_vec))
        all_scores.append(score)

        # Ground truth: genuine if subject + session match
        is_genuine = int(test_id == train_id)
        all_labels.append(is_genuine)

# === Normalize similarity scores to [0, 1]
scores = np.array(all_scores)
labels = np.array(all_labels)
scores = (scores - scores.min()) / (scores.max() - scores.min())
# === Toggle SMOTE ===
use_smote = True  # Set True if you want to apply SMO
# === Apply SMOTE (optional)
if use_smote:
    smote = SMOTE(random_state=42)
    scores_2d = scores.reshape(-1, 1)  # Reshape to 2D: (n_samples, 1)
    scores_2d, labels = smote.fit_resample(scores_2d, labels)
    scores = scores_2d.ravel()  # Flatten back to 1D for thresholding
    print("🧪 After SMOTE label distribution:", Counter(labels))
# === Threshold Sweeping to Find Best F1 Score
best_f1 = best_thresh = best_prec = best_rec = 0

for t in np.linspace(0, 1, 1000):
    preds = (scores >= t).astype(int)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        best_prec = precision
        best_rec = recall

# === Final Classification at Optimal Threshold
final_preds = (scores >= best_thresh).astype(int)
accuracy = accuracy_score(labels, final_preds)

# === Print Summary Report
print("🔍 Summary (Session-Sensitive Verification — Strategy 1: Fused Fingers)")
print("📎 Feature: LBP((8,1),(16,1),(8,2)) + (2D)^2PCA")
print(f"📍 Optimal Threshold  : {best_thresh:.3f}")
print(f"✔️ Accuracy           : {accuracy * 100:.2f}%")
print(f"✔️ Precision (PR)     : {best_prec * 100:.2f}%")
print(f"✔️ Recall (RC)        : {best_rec * 100:.2f}%")
print(f"✔️ F1 Score (F1)      : {best_f1 * 100:.2f}%")


Session Independent R5

In [ ]:
import numpy as np
from collections import defaultdict

# === Configuration ===
ranks = [1, 5]
rank_correct = defaultdict(int)
total_tests = len(test_labels)

print("📊 Calculating Session-Independent CMC (Rank-1 & Rank-5) — Strategy 1: Fused Fingers...")

for i in range(total_tests):
    proj_test = test_lbp_features[i]
    test_label = test_labels[i]

    # ✅ Extract subject only (ignore session for session-independent evaluation)
    test_parts = test_label.split('_')
    test_subject = test_parts[0]
    test_id = test_subject  # No session in ID

    # Compute Manhattan distances
    distances = np.sum(np.abs(train_lbp_features - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    matched = False
    for r in range(1, max(ranks) + 1):
        candidate_label = train_labels[sorted_indices[r - 1]]
        parts = candidate_label.split('_')
        candidate_subject = parts[0]
        candidate_id = candidate_subject  # No session in ID

        if candidate_id == test_id and not matched:
            for k in ranks:
                if r <= k:
                    rank_correct[k] += 1
            matched = True

# === Final CMC Results
for k in ranks:
    accuracy = (rank_correct[k] / total_tests) * 100
    print(f"🎯 Rank-{k} Accuracy (Subject only): {accuracy:.2f}%")


Session Independent CMC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ✅ Helper function: extract subject only (ignore session and finger)
def extract_subject(label):
    return label.split('_')[0]  # subject ID only

# === CONFIGURATION ===
max_rank = 100
rank_correct = np.zeros(max_rank)
total_tests = len(test_lbp_features)

print("📊 Calculating Session-Independent CMC Curve (Matching by Subject Only)...")

for i in range(total_tests):
    proj_test = test_lbp_features[i]
    true_label = test_labels[i]
    true_subject = extract_subject(true_label)

    # Compute Manhattan distances to all training samples
    distances = np.sum(np.abs(train_lbp_features - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    # Find the first correct match
    for r in range(max_rank):
        candidate_label = train_labels[sorted_indices[r]]
        candidate_subject = extract_subject(candidate_label)

        if candidate_subject == true_subject:
            rank_correct[r:] += 1
            break

# ✅ Normalize to percentage
cmc_curve = (rank_correct / total_tests) * 100

# ✅ Plotting the CMC curve
plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, max_rank + 1), cmc_curve, label="Session-Independent CMC", linewidth=2)
plt.xlabel("Rank")
plt.ylabel("Identification Accuracy (%)")
plt.title("Session-Independent CMC Curve — LBP$_{\\mathrm{RIU2}}$((8,1), (16,1), (8,2)) (Strategy 1, Protocol 1)")
plt.grid(True)
plt.xticks(np.arange(0, max_rank + 1, 10))
plt.legend()
plt.tight_layout()
plt.show()

# ✅ Print key rank accuracies
print(f"🎯 Rank-1 Accuracy   : {cmc_curve[0]:.2f}%")
print(f"🎯 Rank-5 Accuracy   : {cmc_curve[4]:.2f}%")
print(f"🎯 Rank-10 Accuracy  : {cmc_curve[9]:.2f}%")
print(f"🎯 Rank-100 Accuracy : {cmc_curve[99]:.2f}%")


Session Independent Precision, Recall, F1, Accuracy

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# === Initialize lists
all_scores = []
all_labels = []

# === Pairwise score computation for Session-Independent Verification (Strategy 1: Fused Fingers)
for test_idx in range(len(test_lbp_features)):
    test_vec = test_lbp_features[test_idx]
    test_label = test_labels[test_idx]
    test_parts = test_label.split('_')
    test_subject = test_parts[0]  # ✅ Use only subject
    test_id = test_subject

    for train_idx in range(len(train_lbp_features)):
        train_vec = train_lbp_features[train_idx]
        train_label = train_labels[train_idx]
        train_parts = train_label.split('_')
        train_subject = train_parts[0]  # ✅ Use only subject
        train_id = train_subject

        # Skip self-comparison (optional but recommended)
        if test_label == train_label:
            continue

        # Similarity score: negative Manhattan distance
        score = -np.sum(np.abs(test_vec - train_vec))
        all_scores.append(score)

        # Ground truth: genuine if subject matches (session ignored)
        is_genuine = int(test_id == train_id)
        all_labels.append(is_genuine)

# === Normalize similarity scores to [0, 1]
scores = np.array(all_scores)
labels = np.array(all_labels)
scores = (scores - scores.min()) / (scores.max() - scores.min())
# === Toggle SMOTE ===
use_smote = True  # Set True if you want to apply SMO
# === Apply SMOTE (optional)
if use_smote:
    smote = SMOTE(random_state=42)
    scores_2d = scores.reshape(-1, 1)  # Reshape to 2D: (n_samples, 1)
    scores_2d, labels = smote.fit_resample(scores_2d, labels)
    scores = scores_2d.ravel()  # Flatten back to 1D for thresholding
    print("🧪 After SMOTE label distribution:", Counter(labels))
# === Threshold Sweeping to Find Best F1 Score
best_f1 = best_thresh = best_prec = best_rec = 0

for t in np.linspace(0, 1, 1000):
    preds = (scores >= t).astype(int)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        best_prec = precision
        best_rec = recall

# === Final Classification at Optimal Threshold
final_preds = (scores >= best_thresh).astype(int)
accuracy = accuracy_score(labels, final_preds)

# === Print Summary Report
print("🔍 Summary (Session-Independent Verification — Strategy 1: Fused Fingers)")
print("📎 Feature: LBP((8,1),(16,1),(8,2)) + (2D)^2PCA")
print(f"📍 Optimal Threshold  : {best_thresh:.3f}")
print(f"✔️ Accuracy           : {accuracy * 100:.2f}%")
print(f"✔️ Precision (PR)     : {best_prec * 100:.2f}%")
print(f"✔️ Recall (RC)        : {best_rec * 100:.2f}%")
print(f"✔️ F1 Score (F1)      : {best_f1 * 100:.2f}%")
